# Example showing routing pattern

A llm request that can route the request to different tool call

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import requests
import json
import logging

# Load all environment variables from .env file
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger(__name__)

In [2]:
# Make a tool

def fetch_temperature(lat: float, long: float) -> float:
    print(f"Calling Weather API for lat={lat}, lon={long}...")
    base_url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": lat, "longitude": long, "current": "temperature_2m"}
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
    return data["current"]["temperature_2m"]

def retrieve_from_kb(question: str) -> dict:
    """Retrieves information from the Educative knowledge base."""
    print(f"Calling Knowledge Base for question: '{question}'...")
    with open("assets/educative_kb.json", "r") as f:
        return json.load(f)

In [3]:
# Make a tool registry

master_tool_registry = [
    {
        "type": "function",
        "function": {
            "name": "fetch_temperature",
            "description": "Return the current temperature (°C) for a given location by its coordinates.",
            "parameters": {
                "type": "object",
                "properties": {
                    "lat": {"type": "number", "description": "The latitude of the location."},
                    "lon": {"type": "number", "description": "The longitude of the location."}
                },
                "required": ["lat", "lon"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "retrieve_from_kb",
            "description": "Answer questions about Educative courses and content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "The user's question about Educative."}
                },
                "required": ["question"]
            }
        }
    }
]

In [4]:
# Given a tool name and arguments, execute the tool and return the result
def execute_tool(tool_name: str, arguments: dict) -> dict:
    """Executes a tool by its name with the provided arguments."""
    logger.info(f"Executing tool: {tool_name} with arguments: {arguments} with argument type: {type(arguments)}")
    if tool_name == "fetch_temperature":
        logger.info(f"Calling fetch_temperature with lat={arguments['lat']} and lon={arguments['lon']}")
        result = fetch_temperature(arguments["lat"], arguments["lon"])
        return {"temperature": result}
    elif tool_name == "retrieve_from_kb":
        logger.info(f"Calling retrieve_from_kb with question: {arguments['question']}")
        result = retrieve_from_kb(arguments["question"])
        return {"kb_result": result}
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

In [5]:
# Routing function that takes user input, determines which tool to call, and returns the tool's output

def route_user_input(user_input: str) -> dict:

    llm_message = [
            {
                "role": "system", 
                "content": """You are a helpful assistant with access to tools. "
                    "For the 'fetch_temperature' tool, if the user provides a location name "
                    "but not coordinates, **use your own general knowledge to determine the latitude and "
                    "longitude, then call the function with those deduced values.** "
                    "If you are unsure or the location is ambiguous, ask the user for clarification."""
            },
            {
                "role": "user", 
                "content": user_input
            }
        ]

    client = OpenAI()
    llm_request = client.chat.completions.create(
        model="gpt-4.1",
        messages=llm_message,
        tools=master_tool_registry
    )

    response_mesage = llm_request.choices[0].message
    logger.info(f"LLM response received: {response_mesage}")
    if response_mesage.tool_calls:
        logger.info(f"Tool call detected: {response_mesage.tool_calls[0].function.name} with arguments {response_mesage.tool_calls[0].function.arguments}")

        llm_message.append(response_mesage)
        tool_execution_result = execute_tool(response_mesage.tool_calls[0].function.name, json.loads(response_mesage.tool_calls[0].function.arguments))
        logger.info(f"Tool execution result: {tool_execution_result}")

        llm_message.append(
            {
                "role": "tool",
                "tool_call_id": response_mesage.tool_calls[0].id,
                "content": json.dumps(tool_execution_result)
            }
        )
    else:
        logger.info("No tool call detected. Returning LLM response.")
        llm_message.append(
            {
                "role": "assistant",
                "content": "I am not able to determine the appropriate tool to call for this query."
            }
        )

    final_llm_request = client.chat.completions.create(
        model="gpt-4.1",
        messages=llm_message
    )

    message_content = final_llm_request.choices[0].message.content
    client.close()

    return message_content



        

In [6]:
route_user_input("Can you check how hot it is in Lucknow, the geographical coordinates of Lucknow are 26.8467 degrees North latitude and 80.9462 degree East longitude?")


2026-05-20 15:06:13 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-20 15:06:13 - INFO - LLM response received: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_WaeixDiWd96ilgjicbO7cj3C', function=Function(arguments='{"lat":26.8467,"lon":80.9462}', name='fetch_temperature'), type='function')])
2026-05-20 15:06:13 - INFO - Tool call detected: fetch_temperature with arguments {"lat":26.8467,"lon":80.9462}
2026-05-20 15:06:13 - INFO - Executing tool: fetch_temperature with arguments: {'lat': 26.8467, 'lon': 80.9462} with argument type: <class 'dict'>
2026-05-20 15:06:13 - INFO - Calling fetch_temperature with lat=26.8467 and lon=80.9462


Calling Weather API for lat=26.8467, lon=80.9462...


2026-05-20 15:06:14 - INFO - Tool execution result: {'temperature': 43.4}
2026-05-20 15:06:16 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


"The current temperature in Lucknow (26.8467°N, 80.9462°E) is 43.4°C. It's quite hot there right now! If you need more weather details, let me know."

In [7]:
route_user_input("What new AI course is Educative releasing?")

2026-05-20 15:06:17 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-20 15:06:17 - INFO - LLM response received: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_MfGnjKwrNzNeqfQ6Fs7bqw2d', function=Function(arguments='{"question":"What new AI course is Educative releasing?"}', name='retrieve_from_kb'), type='function')])
2026-05-20 15:06:17 - INFO - Tool call detected: retrieve_from_kb with arguments {"question":"What new AI course is Educative releasing?"}
2026-05-20 15:06:17 - INFO - Executing tool: retrieve_from_kb with arguments: {'question': 'What new AI course is Educative releasing?'} with argument type: <class 'dict'>
2026-05-20 15:06:17 - INFO - Calling retrieve_from_kb with question: What new AI course is Educative releasing?
2026-05-20 15:06:17 - INFO - Tool execution result: {'kb_result': {'records': [{

Calling Knowledge Base for question: 'What new AI course is Educative releasing?'...


2026-05-20 15:06:18 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


"Educative is actively developing a new course on Agentic System Design. This course will teach how to build AI agents that can plan, reason, and act using memory, tools, and retrieval. It's part of a broader AI initiative that Educative is launching this year.\n\nIf you're interested, Educative also offers related AI courses such as:\n\n- The MCP (Model Context Protocol) course, which covers how AI systems manage prompts, resources, and tools.\n- LlamaStack, which is a hands-on course for building a full-stack AI application using open-source models.\n\nWould you like more information about any of these upcoming courses?"

In [ ]:
route_user_input("Can you write me a short poem about a robot?")

2026-05-20 15:06:20 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-20 15:06:20 - INFO - LLM response received: ChatCompletionMessage(content='Of course! Here’s a short poem about a robot:\n\nIn a world of wires and steel,\nA robot dreams and starts to feel.\nCircuits hum and lights aglow,\nSecrets only robots know.\n\nMetal hands with gentle touch,\nGears that turn, but not too much.\nBinary heart, so brave and true—\nIf I were a robot, I’d want to be you.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
2026-05-20 15:06:20 - INFO - No tool call detected. Returning LLM response.
